# Practice Day 6

##### Demographics — reshaping & pivoting
##### UNION, pivot_table, melt, crosstab

##### Q1 UNION ALL demographic_data.csv and demographic_data_2.xlsx into one DataFrame. How many total countries are there now?
##### Q2 Join income_group.xlsx to the combined DataFrame on Income Group. Inspect the result — what issue do you notice with the income range columns?
##### Q3 Create a pivot_table: rows = Income Group, values = avg Birth rate and avg Internet users.
##### Q4 Use crosstab to count how many countries fall into each Income Group. Normalize to show row percentages.
##### Q5 Melt the combined DataFrame so that Birth rate and Internet users become rows (long format) with a new "indicator" column and "value" column.

In [3]:
import pandas as pd
import numpy as np
df_demo1=pd.read_csv(r"C:\Users\user\Desktop\Python1\DataSample\demographic_data.csv")
df_demo2=pd.read_excel(r"C:\Users\user\Desktop\Python1\DataSample\demographic_data_2.xlsx")
df_income=pd.read_excel(r"C:\Users\user\Desktop\Python1\DataSample\income_group.xlsx")
print(df_demo1.head())
print(df_demo2.head())
print(df_income.head())

           Country Name Country Code  Birth rate  Internet users  \
0                 Aruba          ABW      10.244            78.9   
1           Afghanistan          AFG      35.253             5.9   
2                Angola          AGO      45.985            19.1   
3               Albania          ALB      12.877            57.2   
4  United Arab Emirates          ARE      11.044            88.0   

          Income Group  
0          High income  
1           Low income  
2  Upper middle income  
3  Upper middle income  
4          High income  
    Country Name Country Code  Birth rate  Internet users         Income Group
0     Air Nomads          ANM        12.3         96.5468          High income
1    Water Tribe          WTB        13.2         95.3000           Low income
2  Earth Kingdom          EKD        15.4         95.0534  Upper middle income
3    Fire Nation          FNT        16.7         94.7836  Lower middle income
          Income Group Income (updated)  Incom

In [12]:
# Q1 UNION ALL demographic_data.csv and demographic_data_2.xlsx into one DataFrame. How many total countries are there now?
df = pd.concat([df_demo1, df_demo2], ignore_index = True)
print('Number of total countries:', df['Country Name'].nunique())

Number of total countries: 199


In [ ]:
# Q2 Join income_group.xlsx to the combined DataFrame on Income Group. 

df_q2 = pd.merge(df, df_income, on = 'Income Group')
df_q2.head()

,Country Name,Country Code,Birth rate,Internet users,Income Group,Income (updated),Income (2020)
0,Aruba,ABW,10.244,78.9,High income,"> 12,695","> 12,535"
1,Afghanistan,AFG,35.253,5.9,Low income,"< 1,046","< 1,035"
2,Angola,AGO,45.985,19.1,Upper middle income,"4,096 -12,695","4,046 -12,535"
3,Albania,ALB,12.877,57.2,Upper middle income,"4,096 -12,695","4,046 -12,535"
4,United Arab Emirates,ARE,11.044,88.0,High income,"> 12,695","> 12,535"


In [ ]:
# Inspect the result — what issue do you notice with the income range columns?
df_q2.groupby(['Income Group', 'Income (updated)', 'Income (2020)']).count()

,,,Country Name,Country Code,Birth rate,Internet users
Income Group,Income (updated),Income (2020),,,,
High income,"> 12,695","> 12,535",68,68,68,68
Low income,"< 1,046","< 1,035",31,31,31,31
Lower middle income,"1,046 – 4,095","1,035 – 4,045",51,51,51,51
Upper middle income,"4,096 -12,695","4,046 -12,535",49,49,49,49


In [ ]:
# no country in Super Rich income group and there seems to be some overlapping between High Income and Super High Income criteria

In [28]:
# Q3 Create a pivot_table: rows = Income Group, values = avg Birth rate and avg Internet users.

pivot = pd.pivot_table(
    df_q2,
    values=['Birth rate','Internet users'],
    index='Income Group',          # Rows
    #columns=,       # Columns
    aggfunc=['mean'],           # Aggregation (sum, mean, count...)
    fill_value=0,            # Fill NaN with 0
    margins=True,            # Add row/column totals (like WITH ROLLUP)
    margins_name='Total'
)
pivot

mean               
                    Birth rate Internet users
Income Group                                 
High income          12.746765      74.559848
Low income           36.462839       8.869355
Lower middle income  26.120725      23.786331
Upper middle income  18.672469      41.397410
Total                21.327819      43.148722

In [ ]:
# Q4 Use crosstab to count how many countries fall into each Income Group. 
# Normalize to show row percentages.

In [46]:
pd.crosstab(df_q2['Income Group'], columns='count', normalize=True)

col_0,count
Income Group,
High income,0.341709
Low income,0.155779
Lower middle income,0.256281
Upper middle income,0.246231


In [49]:
# vs the more typical use:
pd.crosstab(df_q2['Income Group'], df_q2['Country Code']).sum(axis=1)

Income Group
High income            68
Low income             31
Lower middle income    51
Upper middle income    49
dtype: int64

In [53]:
df_q2['Income Group'].value_counts(normalize=True).to_frame()

,proportion
Income Group,
High income,0.341709
Lower middle income,0.256281
Upper middle income,0.246231
Low income,0.155779


In [50]:
pd.crosstab(df_q2['Income Group'], df_q2['Income (updated)'], normalize='index')

Income (updated),"1,046 – 4,095","4,096 -12,695","< 1,046","> 12,695"
Income Group,,,,
High income,0.0,0.0,0.0,1.0
Low income,0.0,0.0,1.0,0.0
Lower middle income,1.0,0.0,0.0,0.0
Upper middle income,0.0,1.0,0.0,0.0


In [ ]:
# Q5 Melt the combined DataFrame so that Birth rate and Internet users become rows (long format) 
# with a new "indicator" column and "value" column.

In [36]:
long_df_q2 = pd.melt(
    df_q2,
    id_vars=['Country Name', 'Country Code',
       'Income Group', 'Income (updated)', 'Income (2020)'],          # Columns to keep as-is
    value_vars=['Birth rate','Internet users'], # Columns to melt
    var_name= 'indicator',              # Name for the new 'variable' column
    value_name='value'           # Name for the new 'value' column
)
long_df_q2

,Country Name,Country Code,Income Group,Income (updated),Income (2020),indicator,value
0,Aruba,ABW,High income,"> 12,695","> 12,535",Birth rate,10.2440
1,Afghanistan,AFG,Low income,"< 1,046","< 1,035",Birth rate,35.2530
2,Angola,AGO,Upper middle income,"4,096 -12,695","4,046 -12,535",Birth rate,45.9850
3,Albania,ALB,Upper middle income,"4,096 -12,695","4,046 -12,535",Birth rate,12.8770
4,United Arab Emirates,ARE,High income,"> 12,695","> 12,535",Birth rate,11.0440
...,...,...,...,...,...,...,...
393,Zimbabwe,ZWE,Low income,"< 1,046","< 1,035",Internet users,18.5000
394,Air Nomads,ANM,High income,"> 12,695","> 12,535",Internet users,96.5468
395,Water Tribe,WTB,Low income,"< 1,046","< 1,035",Internet users,95.3000
396,Earth Kingdom,EKD,Upper middle income,"4,096 -12,695","4,046 -12,535",Internet users,95.0534
